In [1]:
import cellxgene_census

In [2]:
import pandas as pd
from typing import Optional

In [57]:
# census_live_query.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, List, Dict, Any
from collections import defaultdict

import cellxgene_census as cg

In [3]:
import tiledbsoma

print(tiledbsoma.__version__)

1.17.1


In [4]:
import os
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_dir = notebook_dir.parent
data_dir = notebook_dir / "data/"

In [14]:
with cellxgene_census.open_soma(census_version="2025-01-30") as census:
    # Access the obs "DataFrame"
    obs = census["census_data"]["homo_sapiens"].obs
    # Get column names
    obs_columns = obs.schema.names
    print(obs_columns)

['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars']


In [17]:
with cellxgene_census.open_soma(census_version="latest") as census:
    # Access the obs "DataFrame"
    obs = census["census_data"]["homo_sapiens"].obs
    # Get column names
    obs_columns = obs.schema.names
    print(obs_columns)

['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars']


In [ ]:
census.get_obs

In [7]:
obs

<DataFrame 's3://cellxgene-census-public-us-west-2/cell-census/2025-08-18/soma/census_data/homo_sapiens/obs' (CLOSED for 'r')>

In [122]:
# ---------------------------
# Query schema and utilities
# ---------------------------


@dataclass
class CensusQuery:
    """Criteria for live filtering of CELLxGENE Census obs.
    Provide labels (NOT ontology IDs) for tissue/disease/etc. that match the obs columns.
    Handle ontology expansion/synonyms before calling this.
    """

    species: str = "homo_sapiens"  # "homo_sapiens" | "mus_musculus"
    tissue_general: Optional[List[str]] = None
    tissue: Optional[List[str]] = None
    disease: Optional[List[str]] = None
    development_stage: Optional[List[str]] = None
    sex: Optional[List[str]] = None
    assay: Optional[List[str]] = None
    is_primary_data: Optional[bool] = True  # Prefer primary by default
    columns_extra: Optional[List[str]] = (
        None  # e.g., ["cell_type"] if you plan to post-aggregate cell types
    )
    # Output controls
    include_cell_type_counts: bool = False
    top_k_cell_types: int = 15  # Only used if include_cell_type_counts=True

    def needed_columns(self) -> List[str]:
        base = [
            "dataset_id",
            "donor_id",
            "tissue_general",
            "tissue",
            "disease",
            "development_stage",
            "sex",
            "assay",
            "is_primary_data",
        ]
        if self.include_cell_type_counts:
            base.append("cell_type")
        if self.columns_extra:
            for c in self.columns_extra:
                if c not in base:
                    base.append(c)
        return base


def _normalize_species(species: str) -> str:
    s = species.strip().lower().replace(" ", "_")
    if s in {"human", "homo", "h sapiens", "h_sapiens", "homo sapiens"}:
        return "homo_sapiens"
    if s in {"mouse", "mice", "mmusculus", "m_musculus", "mus musculus"}:
        return "mus_musculus"
    # assume user passed canonical
    return s


def _or_equals(col: str, values: Iterable[str]) -> str:
    # Build (col == "v1" or col == "v2" ...)
    safe_vals = [str(v).replace('"', '\\"') for v in values if v is not None]
    if not safe_vals:
        return ""
    parts = [f'{col} == "{v}"' for v in safe_vals]
    return "(" + " or ".join(parts) + ")"


def _build_value_filter(q: CensusQuery) -> str:
    clauses = []

    if q.tissue_general:
        clauses.append(_or_equals("tissue_general", q.tissue_general))
    if q.tissue:
        clauses.append(_or_equals("tissue", q.tissue))
    if q.disease:
        clauses.append(_or_equals("disease", q.disease))
    if q.development_stage:
        clauses.append(_or_equals("development_stage", q.development_stage))
    if q.sex:
        clauses.append(_or_equals("sex", q.sex))
    if q.assay:
        clauses.append(_or_equals("assay", q.assay))
    if q.is_primary_data is not None:
        clauses.append(f"is_primary_data == {bool(q.is_primary_data)}")

    # Filter string for SOMA: join non-empty clauses with AND
    clauses = [c for c in clauses if c]
    return " and ".join(clauses) if clauses else ""


# ---------------------------
# Main live query function
# ---------------------------


def query_cellxgene_census_live(
    query: CensusQuery,
    census_version: str = "latest",
    enrich_metadata: bool = False,
    max_results: Optional[int] = None,
) -> pd.DataFrame:
    """Live query against CELLxGENE Census SOMA obs using value_filter, streaming aggregation.

    Returns a DataFrame with one row per dataset_id and columns:
      ['dataset_id', 'n_cells', 'n_donors', 'assays', 'tissues_general', 'tissues',
       'diseases', 'development_stages', 'sexes', 'is_primary_data',
       'cell_type_topK' (optional), 'census_version', ... + optional dataset metadata]

    Args:
        query: CensusQuery with your criteria.
        census_version: 'latest' or a pinned version.
        enrich_metadata: join dataset title/collection info from get_datasets().
        max_results: if provided, truncates to top-N by n_cells.
    """
    org = _normalize_species(query.species)
    value_filter = _build_value_filter(query)
    cols = query.needed_columns()

    # Aggregation holders
    n_cells = defaultdict(int)
    donors = defaultdict(set)
    assays = defaultdict(set)
    tissues_general = defaultdict(set)
    tissues = defaultdict(set)
    diseases = defaultdict(set)
    devstages = defaultdict(set)
    sexes = defaultdict(set)
    primary_flags = defaultdict(list)
    celltype_counts: Dict[str, Dict[str, int]] = defaultdict(lambda: defaultdict(int))

    with cg.open_soma(census_version=census_version) as census:
        if org not in census["census_data"]:
            raise ValueError(f"Organism '{org}' not found in this census version.")
        obs = census["census_data"][org].obs

        reader = obs.read(
            value_filter=value_filter if value_filter else None,
            column_names=cols,
        )

        # Stream over arrow Tables to stay memory-friendly

        for tbl in reader:
            df = tbl.to_pandas(types_mapper=None)
            # Update aggregations
            for ds_id, sub in df.groupby("dataset_id", dropna=True):
                n = len(sub)
                if n == 0:  # shouldn't happen, but be safe
                    continue
                n_cells[ds_id] += n
                donors[ds_id].update(x for x in sub["donor_id"].dropna().unique())
                assays[ds_id].update(x for x in sub["assay"].dropna().unique())
                tissues_general[ds_id].update(x for x in sub["tissue_general"].dropna().unique())
                tissues[ds_id].update(x for x in sub["tissue"].dropna().unique())
                diseases[ds_id].update(x for x in sub["disease"].dropna().unique())
                devstages[ds_id].update(x for x in sub["development_stage"].dropna().unique())
                sexes[ds_id].update(x for x in sub["sex"].dropna().unique())
                primary_flags[ds_id].extend(sub["is_primary_data"].dropna().astype(bool).tolist())

                if query.include_cell_type_counts and "cell_type" in sub.columns:
                    ct_counts = sub["cell_type"].value_counts(dropna=True).to_dict()
                    for ct, c in ct_counts.items():
                        celltype_counts[ds_id][ct] += int(c)

    # Materialize per-dataset rows
    rows: List[Dict[str, Any]] = []
    for ds_id in n_cells.keys():
        row = {
            "dataset_id": ds_id,
            "n_cells": n_cells[ds_id],
            "n_donors": len(donors[ds_id]) if donors[ds_id] else 0,
            "assays": sorted(list(assays[ds_id])) if assays[ds_id] else [],
            "tissues_general": sorted(list(tissues_general[ds_id]))
            if tissues_general[ds_id]
            else [],
            "tissues": sorted(list(tissues[ds_id])) if tissues[ds_id] else [],
            "diseases": sorted(list(diseases[ds_id])) if diseases[ds_id] else [],
            "development_stages": sorted(list(devstages[ds_id])) if devstages[ds_id] else [],
            "sexes": sorted(list(sexes[ds_id])) if sexes[ds_id] else [],
            "is_primary_data": _mode_bool(primary_flags[ds_id]),
            "census_version": census_version,
        }
        if query.include_cell_type_counts:
            # keep top-K as list of tuples
            topK = sorted(celltype_counts[ds_id].items(), key=lambda kv: kv[1], reverse=True)[
                : query.top_k_cell_types
            ]
            row["cell_type_topK"] = topK
        rows.append(row)

    if not rows:
        return pd.DataFrame(
            columns=[
                "dataset_id",
                "n_cells",
                "n_donors",
                "assays",
                "tissues_general",
                "tissues",
                "diseases",
                "development_stages",
                "sexes",
                "is_primary_data",
                "census_version",
                *(["cell_type_topK"] if query.include_cell_type_counts else []),
            ]
        )

    out = pd.DataFrame(rows).sort_values("n_cells", ascending=False, ignore_index=True)
    # Drop zero-cell datasets (if any slipped through)
    out = out[out["n_cells"] > 0].sort_values("n_cells", ascending=False, ignore_index=True)

    # Optional: limit to top-N by size before enrichment
    if max_results is not None and max_results > 0:
        out = out.head(max_results).reset_index(drop=True)

    # Enrich with dataset/collection metadata
    if enrich_metadata:
        try:
            meta = cg.get_datasets(census_version=census_version)  # returns a DataFrame
            # Keep a few useful columns if present
            keep_cols = [
                "dataset_id",
                "dataset_title",
                "dataset_asset_h5ad_uri",
                "collection_id",
                "collection_name",
                "collection_doi",
                "collection_is_published",
                "dataset_published_at",
                "collection_url",
                "dataset_url",
                "explorer_url",
                "reference_organism",
            ]
            meta = meta[[c for c in keep_cols if c in meta.columns]].drop_duplicates("dataset_id")
            out = out.merge(meta, on="dataset_id", how="left")
        except Exception as e:
            # Non-fatal if metadata fetch fails
            out["metadata_error"] = str(e)

    return out


def _mode_bool(vals: List[bool]) -> Optional[bool]:
    if not vals:
        return None
    # Assume field is constant per dataset; fallback to majority vote
    trues = sum(1 for v in vals if v)
    falses = len(vals) - trues
    if trues == falses:
        return None
    return trues > falses

In [149]:
q = CensusQuery(
    species="homo_sapiens",
    tissue_general=["heart"],  # provide ontology-expanded labels as needed
    disease=["normal"],  # include synonyms if you maintain them
    # development_stage= ["fetal stage", "early fetal stage", "late fetal stage"],
    assay=["10x 3' v3", "10x 3' v2", "10x 5' v1", "Smart-seq2"],
    is_primary_data=True,
    include_cell_type_counts=False,
    top_k_cell_types=10,
)
q

CensusQuery(species='homo_sapiens', tissue_general=['heart'], tissue=None, disease=['normal'], development_stage=None, sex=None, assay=["10x 3' v3", "10x 3' v2", "10x 5' v1", 'Smart-seq2'], is_primary_data=True, columns_extra=None, include_cell_type_counts=False, top_k_cell_types=10)

In [150]:
cols = q.needed_columns()
cols

['dataset_id',
 'donor_id',
 'tissue_general',
 'tissue',
 'disease',
 'development_stage',
 'sex',
 'assay',
 'is_primary_data']

In [151]:
org = _normalize_species(q.species)
org

'homo_sapiens'

In [152]:
value_filter = _build_value_filter(q)

In [153]:
value_filter

'(tissue_general == "heart") and (disease == "normal") and (assay == "10x 3\' v3" or assay == "10x 3\' v2" or assay == "10x 5\' v1" or assay == "Smart-seq2") and is_primary_data == True'

In [154]:
with cg.open_soma(census_version="latest") as census:
    obs = census["census_data"][org].obs

    reader = obs.read(value_filter=value_filter, column_names=cols)

    for tbl in reader:
        df = tbl.to_pandas(types_mapper=None)

In [155]:
df

,dataset_id,donor_id,tissue_general,tissue,disease,development_stage,sex,assay,is_primary_data
0,87a06a2a-16c6-42f7-af1b-529fad431051,Control_donor_1,heart,left atrium auricular region,normal,fifth decade stage,male,10x 3' v3,True
1,87a06a2a-16c6-42f7-af1b-529fad431051,Control_donor_1,heart,left atrium auricular region,normal,fifth decade stage,male,10x 3' v3,True
2,87a06a2a-16c6-42f7-af1b-529fad431051,Control_donor_1,heart,left atrium auricular region,normal,fifth decade stage,male,10x 3' v3,True
3,87a06a2a-16c6-42f7-af1b-529fad431051,Control_donor_1,heart,left atrium auricular region,normal,fifth decade stage,male,10x 3' v3,True
4,87a06a2a-16c6-42f7-af1b-529fad431051,Control_donor_1,heart,left atrium auricular region,normal,fifth decade stage,male,10x 3' v3,True
...,...,...,...,...,...,...,...,...,...
1637114,53d208b0-2cfd-4366-9866-c3c6114081bc,TSP2,heart,cardiac ventricle,normal,61-year-old stage,female,10x 3' v3,True
1637115,53d208b0-2cfd-4366-9866-c3c6114081bc,TSP2,heart,cardiac ventricle,normal,61-year-old stage,female,10x 3' v3,True
1637116,53d208b0-2cfd-4366-9866-c3c6114081bc,TSP2,heart,cardiac ventricle,normal,61-year-old stage,female,10x 3' v3,True
1637117,53d208b0-2cfd-4366-9866-c3c6114081bc,TSP2,heart,cardiac ventricle,normal,61-year-old stage,female,10x 3' v3,True


In [156]:
# Example: human heart, healthy, primary data, sc/snRNA assays
q = CensusQuery(
    species="homo_sapiens",
    tissue_general=["heart"],  # provide ontology-expanded labels as needed
    disease=["normal"],  # include synonyms if you maintain them
    # development_stage= ["fetal", "13 pcw", "12 pcw", "14 pcw"],
    assay=["10x 3' v3", "10x 3' v2", "10x 5' v1", "Smart-seq2", "snRNA-seq"],
    is_primary_data=True,
    include_cell_type_counts=False,
    top_k_cell_types=10,
)

df = query_cellxgene_census_live(
    query=q,
    census_version="latest",
    enrich_metadata=True,
    max_results=50,
)
# Print a compact preview
with pd.option_context("display.max_colwidth", 100):
    print(df.head(1))

                             dataset_id  n_cells  n_donors  \
0  d4e69e01-3ba2-4d6b-a15d-e7048f78f22e   486134        14   

                   assays tissues_general  \
0  [10x 3' v2, 10x 3' v3]         [heart]   

                                                                                               tissues  \
0  [apex of heart, heart left ventricle, heart right ventricle, interventricular septum, left cardi...   

   diseases  \
0  [normal]   

                                                                    development_stages  \
0  [eighth decade stage, fifth decade stage, seventh decade stage, sixth decade stage]   

            sexes  is_primary_data census_version  \
0  [female, male]             True         latest   

                                              metadata_error  
0  module 'cellxgene_census' has no attribute 'get_datasets'  


/var/folders/fs/3n9_hpnx7rv0vvh25klngrk80000gq/T/ipykernel_26477/550668953.py:143: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for ds_id, sub in df.groupby("dataset_id", dropna=True):


In [159]:
df.columns

Index(['dataset_id', 'n_cells', 'n_donors', 'assays', 'tissues_general',
       'tissues', 'diseases', 'development_stages', 'sexes', 'is_primary_data',
       'census_version', 'metadata_error'],
      dtype='object')

In [162]:
df

,dataset_id,n_cells,n_donors,assays,tissues_general,tissues,diseases,development_stages,sexes,is_primary_data,census_version,metadata_error
0,d4e69e01-3ba2-4d6b-a15d-e7048f78f22e,486134,14,"[10x 3' v2, 10x 3' v3]",[heart],"[apex of heart, heart left ventricle, heart ri...",[normal],"[eighth decade stage, fifth decade stage, seve...","[female, male]",True,latest,module 'cellxgene_census' has no attribute 'ge...
1,364bd0c7-f7fd-48ed-99c1-ae26872b1042,437851,32,"[10x 3' v2, 10x 5' v1]",[heart],"[heart left ventricle, heart right ventricle, ...",[normal],"[20-year-old stage, 21-year-old stage, 27-year...","[female, male]",True,latest,module 'cellxgene_census' has no attribute 'ge...
2,f1606894-59df-4794-a37f-baa7c6fb6de1,217167,29,[10x 3' v3],[heart],[right atrium auricular region],[normal],"[eighth decade stage, fifth decade stage, nint...","[female, male]",True,latest,module 'cellxgene_census' has no attribute 'ge...
3,8f4f8502-9170-4ac2-9707-3b6985ebfe5f,146966,31,[10x 3' v3],[heart],[right atrium auricular region],[normal],"[eighth decade stage, fifth decade stage, nint...","[female, male]",True,latest,module 'cellxgene_census' has no attribute 'ge...
4,65badd7a-9262-4fd1-9ce2-eb5dc0ca8039,67246,6,[10x 3' v3],[heart],"[heart left ventricle, heart right ventricle, ...",[normal],"[fifth decade stage, seventh decade stage, six...","[female, male]",True,latest,module 'cellxgene_census' has no attribute 'ge...
5,99e317c0-1fab-4f73-b511-859e752fd1c3,51176,9,[10x 3' v3],[heart],[apical region of left ventricle],[normal],"[10-year-old stage, 14-year-old stage, 19th we...","[female, male]",True,latest,module 'cellxgene_census' has no attribute 'ge...
6,2e9d2f32-4cfb-49b5-b990-cbf4c241214e,49359,3,[10x 3' v3],[heart],"[apex of heart, heart left ventricle, heart ri...",[normal],"[47-year-old stage, 54-year-old stage, 55-year...",[male],True,latest,module 'cellxgene_census' has no attribute 'ge...
7,1c739a3e-c3f5-49d5-98e0-73975e751201,41663,4,[10x 3' v3],[heart],[heart left ventricle],[normal],"[44-year-old stage, 55-year-old stage, 61-year...","[female, male]",True,latest,module 'cellxgene_census' has no attribute 'ge...
8,53d208b0-2cfd-4366-9866-c3c6114081bc,40006,6,"[10x 3' v3, Smart-seq2]",[heart],"[cardiac atrium, cardiac ventricle, coronary a...",[normal],"[45-year-old stage, 56-year-old stage, 59-year...","[female, male]",True,latest,module 'cellxgene_census' has no attribute 'ge...
9,4ed927e9-c099-49af-b8ce-a2652d069333,36574,3,[10x 3' v2],[heart],[anterior wall of left ventricle],[normal],"[eighth decade stage, seventh decade stage, si...","[female, male]",True,latest,module 'cellxgene_census' has no attribute 'ge...


In [161]:
df["development_stages"]

0     [eighth decade stage, fifth decade stage, seve...
1     [20-year-old stage, 21-year-old stage, 27-year...
2     [eighth decade stage, fifth decade stage, nint...
3     [eighth decade stage, fifth decade stage, nint...
4     [fifth decade stage, seventh decade stage, six...
5     [10-year-old stage, 14-year-old stage, 19th we...
6     [47-year-old stage, 54-year-old stage, 55-year...
7     [44-year-old stage, 55-year-old stage, 61-year...
8     [45-year-old stage, 56-year-old stage, 59-year...
9     [eighth decade stage, seventh decade stage, si...
10    [10th week post-fertilization stage, 12th week...
11    [eighth decade stage, fifth decade stage, seve...
12    [38-year-old stage, 39-year-old stage, 46-year...
13                               [seventh decade stage]
Name: development_stages, dtype: object

In [5]:
# collection_id = "d86517f0-fa7e-4266-b82e-a521350d6d36"
# dataset_id = "f7341911-5051-4bae-8b46-9ddfa880084f"

In [6]:
# cellxgene_census.download_source_h5ad("dbb4e1ed-d820-4e83-981f-88ef7eb55a35", str(data_dir / "single_cell_ref.h5ad"), census_version="latest", progress_bar=True)

In [13]:
with cellxgene_census.open_soma(
    census_version="latest", tiledb_config={"py.init_buffer_bytes": 128 * 1024**2}
) as census:
    cxg_ds = census["census_info"]["datasets"].read().concat().to_pandas()
cxg_ds.head(2)

,soma_joinid,citation,collection_id,collection_name,collection_doi,collection_doi_label,dataset_id,dataset_version_id,dataset_title,dataset_h5ad_path,dataset_total_cell_count
0,0,Publication: https://doi.org/10.1016/j.isci.20...,8e880741-bf9a-4c8e-9227-934204631d2a,High Resolution Slide-seqV2 Spatial Transcript...,10.1016/j.isci.2022.104097,Marshall et al. (2022) iScience,4eb29386-de81-452f-b3c0-e00844e8c7fd,f76861bb-becb-4eb7-82fc-782dc96ccc7f,Spatial transcriptomics in mouse: Puck_191112_05,4eb29386-de81-452f-b3c0-e00844e8c7fd.h5ad,10888
1,1,Publication: https://doi.org/10.1016/j.isci.20...,8e880741-bf9a-4c8e-9227-934204631d2a,High Resolution Slide-seqV2 Spatial Transcript...,10.1016/j.isci.2022.104097,Marshall et al. (2022) iScience,78d59e4a-82eb-4a61-a1dc-da974d7ea54b,7d7ec1b6-6e3f-4aaa-9442-4b22f3424396,Spatial transcriptomics in mouse: Puck_191112_08,78d59e4a-82eb-4a61-a1dc-da974d7ea54b.h5ad,10250


In [163]:
import cellxgene_census as cg
import pandas as pd
from collections import Counter

# Which obs fields you care about (add/remove as needed)
CELL_METADATA_FIELDS = [
    "assay",
    "cell_type",
    "development_stage",
    "disease",
    "self_reported_ethnicity",
    "sex",
    "suspension_type",
    "tissue",
    "tissue_general",
    # You can also inspect the *_ontology_term_id variants:
    # "assay_ontology_term_id", "cell_type_ontology_term_id", "development_stage_ontology_term_id",
    # "disease_ontology_term_id", "sex_ontology_term_id", "tissue_ontology_term_id",
    # "tissue_general_ontology_term_id",
]


def _iter_tables(reader):
    """Works across tiledbsoma versions (reader iterable; no context manager needed)."""
    for t in reader:
        yield t
    if hasattr(reader, "close"):
        reader.close()


def census_obs_distinct_counts(
    organism: str,
    fields: List[str] = CELL_METADATA_FIELDS,
    census_version: str = "latest",
    value_filter: Optional[str] = None,
) -> Dict[str, int]:
    """Return {field: number_of_distinct_non_null_values} for obs fields."""
    counts: Dict[str, int] = {}
    with cg.open_soma(census_version=census_version) as census:
        obs = census["census_data"][organism].obs
        present = [f for f in fields if f in obs.schema.names]
        acc = {f: set() for f in present}

        reader = obs.read(value_filter=value_filter or None, column_names=present)
        for tbl in _iter_tables(reader):
            df = tbl.to_pandas(types_mapper=None)
            for f in present:
                if f in df:
                    acc[f].update(df[f].dropna().unique().tolist())

    for f, s in acc.items():
        counts[f] = len(s)
    return counts


def census_obs_value_counts(
    organism: str,
    fields: List[str] = CELL_METADATA_FIELDS,
    census_version: str = "latest",
    value_filter: Optional[str] = None,
    top_per_field: Optional[int] = None,
) -> pd.DataFrame:
    """Return a long DataFrame with per-field value counts under an optional value_filter.
    Columns: ['field', 'value', 'n_cells'].
    """
    with cg.open_soma(census_version=census_version) as census:
        obs = census["census_data"][organism].obs
        present = [f for f in fields if f in obs.schema.names]
        counters = {f: Counter() for f in present}

        reader = obs.read(value_filter=value_filter or None, column_names=present)
        for tbl in _iter_tables(reader):
            df = tbl.to_pandas(types_mapper=None)
            for f in present:
                if f in df:
                    vc = df[f].dropna().value_counts()
                    counters[f].update(vc.to_dict())

    rows = []
    for f, ctr in counters.items():
        items = ctr.most_common(top_per_field) if top_per_field else ctr.items()
        for v, n in items:
            rows.append({"field": f, "value": v, "n_cells": int(n)})
    out = pd.DataFrame(rows).sort_values(
        ["field", "n_cells"], ascending=[True, False], ignore_index=True
    )
    return out

In [164]:
fields = CELL_METADATA_FIELDS
hs_counts = census_obs_distinct_counts("homo_sapiens", fields)
mm_counts = census_obs_distinct_counts("mus_musculus", fields)

summary = pd.DataFrame(
    {
        "Category": fields,
        "Homo sapiens": [hs_counts.get(f, 0) for f in fields],
        "Mus musculus": [mm_counts.get(f, 0) for f in fields],
    }
)
print(summary)

                  Category  Homo sapiens  Mus musculus
0                    assay            29            15
1                cell_type           831           448
2        development_stage           184            59
3                  disease           144            14
4  self_reported_ethnicity            37             1
5                      sex             3             3
6          suspension_type             2             2
7                   tissue           374           101
8           tissue_general            68            36


In [165]:
import cellxgene_census as cg
import pandas as pd


def all_development_stage_values(
    organism: str = "homo_sapiens",
    census_version: str = "latest",
    value_filter: Optional[
        str
    ] = None,  # e.g., 'tissue_general == "heart" and is_primary_data == True'
):
    vals = set()
    with cg.open_soma(census_version=census_version) as census:
        obs = census["census_data"][organism].obs
        reader = obs.read(value_filter=value_filter or None, column_names=["development_stage"])
        for tbl in reader:  # reader is iterable; no context manager
            s = tbl.to_pandas()["development_stage"].dropna().astype(str)
            vals.update(s.unique().tolist())
        if hasattr(reader, "close"):  # some versions expose .close()
            reader.close()
    return sorted(vals)


def development_stage_vocab_with_counts(
    organism: str = "homo_sapiens",
    census_version: str = "latest",
    value_filter: Optional[str] = None,
) -> pd.DataFrame:
    """Returns a DataFrame: development_stage, development_stage_ontology_term_id, n_cells"""
    counts = Counter()
    id_map = {}
    with cg.open_soma(census_version=census_version) as census:
        obs = census["census_data"][organism].obs
        cols = ["development_stage", "development_stage_ontology_term_id"]
        reader = obs.read(value_filter=value_filter or None, column_names=cols)
        for tbl in reader:
            df = tbl.to_pandas()[cols].dropna(subset=["development_stage"])
            counts.update(df["development_stage"].value_counts().to_dict())
            # keep the first seen ID per label (IDs are usually consistent)
            for lbl, oid in df.dropna(subset=["development_stage_ontology_term_id"]).itertuples(
                index=False
            ):
                id_map.setdefault(lbl, oid)
        if hasattr(reader, "close"):
            reader.close()
    rows = [
        {
            "development_stage": k,
            "development_stage_ontology_term_id": id_map.get(k),
            "n_cells": int(v),
        }
        for k, v in counts.items()
    ]
    return pd.DataFrame(rows).sort_values("n_cells", ascending=False, ignore_index=True)

In [166]:
vals = all_development_stage_values("homo_sapiens")
print(len(vals), vals[:15])

# B) Heart-only, primary cells
vf = 'tissue_general == "heart" and is_primary_data == True'
vals_heart = all_development_stage_values("homo_sapiens", value_filter=vf)
print(vals_heart)

184 ['1-month-old stage', '1-year-old stage', '10-month-old stage', '10-year-old stage', '101-year-old stage', '102-year-old stage', '10th week post-fertilization stage', '11-month-old stage', '11-year-old stage', '111-year-old stage', '11th week post-fertilization stage', '12-month-old stage', '12-year-old stage', '12th week post-fertilization stage', '13-month-old stage']
['10-year-old stage', '10th week post-fertilization stage', '11th week post-fertilization stage', '12th week post-fertilization stage', '13th week post-fertilization stage', '14-year-old stage', '15th week post-fertilization stage', '16th week post-fertilization stage', '17th week post-fertilization stage', '19th week post-fertilization stage', '20-year-old stage', '20th week post-fertilization stage', '21-year-old stage', '27-year-old stage', '29-year-old stage', '35-year-old stage', '38-year-old stage', '39-year-old stage', '4-year-old stage', '40-year-old stage', '41-year-old stage', '42-year-old stage', '43-year